# Calibrate

Experiment 04 let every projection read the same fraction of its rows, stored in checkpoint order.
At 30% of the rows the chunk selectors then answered no better than chance, while the paper stays above 45%.

The paper prepares two things offline, on videos it does not evaluate on:
the hot–cold order of the rows of each projection, and how many rows each projection reads.
This notebook computes both for LLaVA-OneVision-Qwen2-0.5B.
[05_02](05_02_run_vlm.ipynb) runs the model with them.

The paper is [VLM in a flash](https://arxiv.org/abs/2511.18692); upstream is its code in `../../vlm-flash`.
The rows per projection follow [TEAL](https://arxiv.org/abs/2408.14690), as in the paper.

In [1]:
from copy     import deepcopy
from pathlib  import Path
from tempfile import TemporaryDirectory
from zipfile  import ZipFile

from huggingface_hub import hf_hub_download
from transformers    import AutoProcessor, DynamicCache, LlavaOnevisionForConditionalGeneration

import sys
import torch

import numpy  as np
import pandas as pd

/home/yabsed/miniconda3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
UPSTREAM = Path("../../vlm-flash")

sys.path.insert(0, str((UPSTREAM / "src").resolve()))
sys.path.insert(0, str((UPSTREAM / "scripts").resolve()))

import vlmflash
from _vlm import encode_video, load_frames

## 1. Model

The model is a vision tower followed by a language model.
It takes a video and a text, and writes a text.
Only the language model's weights are selected.

In [3]:
MODEL_ID       = "llava-hf/llava-onevision-qwen2-0.5b-ov-hf"
MODEL_REVISION = "74dd0bf867a4cda7950c17663794267c60cf4b40"

In [4]:
# Inference only: no gradients below.
torch.set_grad_enabled(False)

device = "cuda" if torch.cuda.is_available() else "cpu"

processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    revision = MODEL_REVISION,
)

net = LlavaOnevisionForConditionalGeneration.from_pretrained(
    MODEL_ID,
    revision = MODEL_REVISION,
    dtype    = torch.float16,
).to(device).eval()

The image processor of type `LlavaOnevisionImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 
Loading weights: 100%|██████████| 716/716 [00:00<00:00, 1224.24it/s, Materializing param=model.vision_tower.vision_model.post_layernorm.weight]                      


In [5]:
# The language model inside the vision-language wrapper.
decoder = net.model.language_model

The language model is a stack of blocks, `decoder.layers`.
The attention and the MLP of a block are built from linear projections.
Every block has seven: `q_proj`, `k_proj`, `v_proj`, and `o_proj` in attention;
`gate_proj`, `up_proj`, and `down_proj` in the MLP.

Each input channel of a projection owns one row of the weight matrix stored on flash,
so choosing channels chooses the rows to read.

In [6]:
# projection name -> layer
projections = {}

for name, layer in decoder.named_modules():
    if name.endswith("_proj"):
        projections[name] = layer

len(projections)

168

## 2. Videos

TempCompass is one of the paper's benchmarks. It asks questions about 410 short videos.
The paper calibrates on 25 of the videos and evaluates on the others.

In [7]:
DATA_ID       = "lmms-lab/TempCompass"
DATA_REVISION = "c8a67d88a3bc6fd4b2f9ae2f9e112668fbe05722"

In [8]:
question_file = hf_hub_download(
    DATA_ID,
    "multi-choice/test-00000-of-00001.parquet",
    repo_type = "dataset",
    revision  = DATA_REVISION,
)

video_file = hf_hub_download(
    DATA_ID,
    "tempcompass_videos.zip",
    repo_type = "dataset",
    revision  = DATA_REVISION,
)

In [9]:
questions = pd.read_parquet(question_file)
archive   = ZipFile(video_file)

Experiment 04 evaluated on 100 randomly chosen videos, and 05_02 evaluates on the same ones.
Calibrate on 25 of the other 310.

In [10]:
VIDEOS      = 100  # Videos that 04 and 05_02 evaluate on
CALIBRATION = 25   # Videos to calibrate on

every_id = questions.video_id.drop_duplicates()

evaluation_ids  = every_id.sample(VIDEOS, random_state=0)
calibration_ids = every_id.drop(evaluation_ids.index).sample(CALIBRATION, random_state=0)

len(calibration_ids)

25

## 3. Video

Sample 32 frames uniformly over a video.

In [11]:
FRAMES = 32  # Frames sampled from each video

def load_video(video_id):
    """Return the sampled frames of one video as images."""

    # The frame loader reads from a file, so unpack the video first.
    with TemporaryDirectory() as folder:
        path = archive.extract(f"videos/{video_id}.mp4", folder)

        # load_frames also returns a flag that is True only for upstream's built-in clip.
        frames, _ = load_frames(path, FRAMES)

    return frames

The vision tower turns every frame into the same number of tokens,
each as wide as the language model's hidden state.

In [12]:
def encode(frames):
    """Return the frame tokens of one video: (frames, 1, tokens per frame, hidden width)."""

    video_tokens = encode_video(net, processor, frames, device)

    return video_tokens.reshape(len(frames), 1, -1, video_tokens.shape[-1])

## 4. Frame appending

The chat template of the model wraps a text and puts a placeholder where the video goes.

In [13]:
def write_prompt(text):
    """Return the chat prompt of one text, with a placeholder for the video."""

    content = [
        {"type" : "video"},
        {"type" : "text", "text" : text},
    ]

    conversation = [
        {"role" : "user", "content" : content},
    ]

    return processor.apply_chat_template(
        conversation,
        add_generation_prompt = True,
    )

The part of the prompt before the placeholder is the prefix. It is the same for every text.

In [14]:
embedding = decoder.get_input_embeddings()

def embed_text(text):
    """Return the token embeddings of a text: (1, tokens, hidden width)."""

    ids = processor.tokenizer(text, return_tensors="pt").input_ids

    return embedding(ids.to(device))

In [15]:
# The placeholder is processor.video_token.
before, _ = write_prompt("").split(processor.video_token)
prefix    = embed_text(before)

The language model appends the prefix, then one frame per forward pass, to its key–value cache.
Calibration stops there: it watches the videos and asks nothing.

In [16]:
def append(cache, tokens):
    """Run the language model on tokens (videos, tokens, hidden width), extending the cache."""

    return decoder(
        inputs_embeds   = tokens,
        past_key_values = cache,
        use_cache       = True,
    )

## 5. Hot–cold order

In a forward pass, a projection with $N$ input channels receives $x \in \mathbb{R}^{T \times N}$,
the activations of $T$ tokens. The importance of channel $i$, and of its weight row, is

$$
v_i = \frac{1}{T}
\sum_{t=1}^{T}|x_{t,i}|.
$$

A row is active in a frame pass when it is in the more important half of the rows of its projection.
Over the frame passes of the calibration videos, count how often each row is active.
The hot–cold order of a projection lists its rows from the most often active to the least.

Upstream's `FrequencyProfiler` counts in every forward pass that runs inside its `with` block.

In [17]:
profiler = vlmflash.FrequencyProfiler(
    decoder,
    include  = r"_proj$",  # The projections collected in section 1
    sparsity = 0.5,        # Active: the more important half
)

In [18]:
# Middle frame of every calibration video, kept for section 6
middle_frames = []

for video_id in calibration_ids:
    frame_tokens = encode(load_video(video_id))
    middle_frames.append(frame_tokens[FRAMES // 2])

    cache = DynamicCache()
    append(cache, prefix)

    # Count in the frame passes only.
    with profiler:
        for tokens in frame_tokens:
            append(cache, tokens)

# Frame passes counted
profiler.num_samples

800

In [19]:
# projection name -> its rows, most often active first
orders = profiler.orders()

Save the orders to `orders.npz`, one array per projection.

In [20]:
arrays = {}

for name, order in orders.items():
    arrays[name] = order.numpy()

np.savez_compressed("orders.npz", **arrays)

## 6. Rows per projection

A condition of 05_02 requests a fraction $f$ of the projection weights.
Experiment 04 let every projection read the fraction $f$ of its rows.
TEAL instead searches every block for the fractions that change the output of the block least.

Number the seven projections of a block $p = 1, \dots, 7$.
Projection $p$ has $N_p$ rows of $d_p$ weights each, and reads a fraction $f_p$ of its rows.
The block then reads the fraction

$$
F = \frac{\sum_p N_p d_p f_p}{\sum_p N_p d_p}
$$

of its projection weights.

In [21]:
BLOCKS = len(decoder.layers)

# names[block, p]: name of projection p of a block
names = np.array(list(projections)).reshape(BLOCKS, 7)

names[0]

array(['layers.0.self_attn.q_proj', 'layers.0.self_attn.k_proj',
       'layers.0.self_attn.v_proj', 'layers.0.self_attn.o_proj',
       'layers.0.mlp.gate_proj', 'layers.0.mlp.up_proj',
       'layers.0.mlp.down_proj'], dtype='<U26')

In [22]:
# The seven projections have the same shapes in every block.
N     = []  # N[p]: rows of projection p
sizes = []  # sizes[p]: weights of projection p

for name in names[0]:
    layer = projections[name]

    N.append(layer.in_features)
    sizes.append(layer.in_features * layer.out_features)

sizes = np.array(sizes)

In [23]:
def block_fractions(fractions):
    """Return F of every block from fractions[block, p]: (blocks,)."""

    return fractions @ sizes / sizes.sum()

During the search a projection at fraction $f_p$ keeps its
$R_p = \max(1, \lfloor f_p N_p \rfloor)$ most important rows of a frame
and zeroes the input channels of the others. Top-R does the same in 05_02.
TEAL itself zeroes single activations below a threshold.

In [24]:
def budget(fraction, rows):
    """Return R: how many of its rows a projection reads at a fraction."""

    return max(1, int(fraction * rows))

In [25]:
class TopRows:

    """Forward pre-hook of one projection: keep the R most important input channels of each frame."""

    def __init__(self, R):
        self.R = R

    def __call__(self, layer, inputs):

        # Importance of every channel in every frame: (frames, N)
        x      = inputs[0]
        values = x.float().abs().mean(dim=1)

        # 1 at the R most important channels of each frame, 0 at the others
        top  = values.topk(self.R, dim=1).indices
        mask = torch.zeros_like(values)
        mask.scatter_(1, top, 1)

        # The returned tuple replaces the inputs of the layer.
        return (x * mask[:, None, :].to(x.dtype),)

The error of a block is $\lVert \hat{y} - y \rVert_2$.
Here $y$ is the output of the block when its projections read every row,
and $\hat{y}$ its output at the fractions under trial.

Every block is measured on its own: its input comes from blocks that read every row.

In [26]:
class Error:

    """Forward hook of one block: measure how far its output is from the dense output.

    The hook then passes the dense output on, so that the next block receives dense input."""

    def __init__(self):
        self.dense = None  # Output with every row, recorded by the first pass
        self.value = None  # Error of the last pass

    def __call__(self, block, inputs, output):

        # First pass
        if self.dense is None:
            self.dense = output
            return

        difference = (output - self.dense).float()
        self.value = float(difference.norm())

        # The returned tensor replaces the output of the block.
        return self.dense

Measure on the middle frame of every calibration video, read right after the prefix.
The 25 frames go through the model as one batch.

In [27]:
# (videos, tokens per frame, hidden width)
calibration_tokens = torch.cat(middle_frames)

# Cache of the prefix, once per video
calibration_cache = DynamicCache()
append(calibration_cache, prefix.expand(CALIBRATION, -1, -1))

calibration_tokens.shape

torch.Size([25, 196, 896])

Hook every projection and every block. Every projection starts with all of its rows.

In [28]:
keepers = []  # keepers[block][p]: hook of projection p of a block
errors  = []  # errors[block]: hook of a block
handles = []  # Handles that remove the hooks

for index, block in enumerate(decoder.layers):
    hooks = []

    for p, name in enumerate(names[index]):
        hook = TopRows(N[p])
        hooks.append(hook)

        handles.append(projections[name].register_forward_pre_hook(hook))

    keepers.append(hooks)

    hook = Error()
    errors.append(hook)

    handles.append(block.register_forward_hook(hook))

In [29]:
def measure(trial):
    """Return the error of every block when its projections read trial[block, p]: (blocks,)."""

    # Rows of every projection
    for index in range(BLOCKS):
        for p in range(7):
            keepers[index][p].R = budget(trial[index, p], N[p])

    # The cache of the prefix stays as it is.
    append(deepcopy(calibration_cache), calibration_tokens)

    values = []

    for hook in errors:
        values.append(hook.value)

    return np.array(values)

In [30]:
# First pass: every row, so every block records its dense output.
append(deepcopy(calibration_cache), calibration_tokens)

# With every row again, no block is off its dense output.
measure(np.ones((BLOCKS, 7))).max()

np.float64(0.0)

The search starts at $f_p = 1$. One step lowers one $f_p$ by

$$
\delta_p = \alpha \, \frac{N_q d_q}{N_p d_p},
$$

where $q$ is `q_proj` and $\alpha = 0.05$ is TEAL's base step.
Every step therefore lowers $F$ by the same amount.

A step tries each of the seven projections and takes the one with the smallest error.
It does not try a projection that would be left without a row.

In [31]:
BASE_STEP = 0.05

# steps[p]: step of projection p. q_proj is projection 0.
steps = BASE_STEP * sizes[0] / sizes

steps.round(4)

array([0.05  , 0.35  , 0.35  , 0.05  , 0.0092, 0.0092, 0.0092])

In [32]:
def lower(fractions):
    """Take one step in every block: lower the projection whose step changes the output least."""

    # trial_errors[block, p]: error of the block when projection p takes its step
    trial_errors = np.full((BLOCKS, 7), np.inf)

    for p in range(7):
        trial        = fractions.copy()
        trial[:, p] -= steps[p]

        # Blocks whose projection p keeps a row after the step
        allowed = trial[:, p] * N[p] >= 1

        # The other blocks do not take the step.
        trial[~allowed, p] = fractions[~allowed, p]

        trial_errors[allowed, p] = measure(trial)[allowed]

    # Every block has a projection left to lower.
    assert np.isfinite(trial_errors).any(axis=1).all()

    for index in range(BLOCKS):
        p = trial_errors[index].argmin()

        fractions[index, p] -= steps[p]

Search down to 30% of the weights. Keep the fractions at each of the paper's levels:
the first ones at which no block reads more than the requested fraction.
The paper's sparsity is $1 - f$.

In [33]:
FRACTIONS = [1.0, 0.9, 0.8, 0.7, 0.6, 0.5, 0.4, 0.3]  # Requested fractions f

# fractions[block, p]: fraction of its rows that projection p of a block reads
fractions = np.ones((BLOCKS, 7))

# requested fraction -> fractions[block, p] at that fraction
found = {}

for fraction in FRACTIONS:

    while block_fractions(fractions).max() > fraction:
        lower(fractions)

    found[fraction] = fractions.copy()

    print(f"{fraction:.0%} found", flush=True)

100% found
90% found
80% found
70% found
60% found
50% found
40% found
30% found


In [34]:
for handle in handles:
    handle.remove()

The fraction of rows of each of the seven projections, averaged over the blocks:

In [35]:
kinds = []

for name in names[0]:
    kinds.append(name.split(".")[-1])

averages = {}

for fraction, levels in found.items():
    averages[fraction] = levels.mean(axis=0)

pd.DataFrame(averages, index=kinds).T.round(2)

,q_proj,k_proj,v_proj,o_proj,gate_proj,up_proj,down_proj
1.0,1.00,1.00,1.00,1.00,1.00,1.00,1.00
0.9,0.56,0.85,0.97,0.69,0.92,0.97,0.91
0.8,0.44,0.77,0.93,0.59,0.80,0.93,0.76
0.7,0.36,0.66,0.87,0.51,0.69,0.87,0.63
0.6,0.31,0.62,0.85,0.45,0.58,0.77,0.51
0.5,0.27,0.59,0.81,0.42,0.47,0.66,0.41
0.4,0.24,0.50,0.78,0.39,0.36,0.54,0.31
0.3,0.23,0.48,0.77,0.35,0.26,0.40,0.22


Save the rows of every projection at every requested fraction to `budgets.csv`.

In [36]:
records = []

for fraction, levels in found.items():
    for index in range(BLOCKS):
        for p in range(7):
            records.append({
                "fraction"   : fraction,
                "projection" : names[index, p],
                "R"          : budget(levels[index, p], N[p]),
            })

budgets = pd.DataFrame(records)
budgets.to_csv("budgets.csv", index=False)